# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arisckm/vigilant-system/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: If a page has high historic impressions (>median) but its trend direction is 'down', flag it for an urgent content refresh.
Reason Code: REFRESH_STALE_DECLINING_TRAFFIC
Action Label: prioritize_refresh

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)

# 1. Load data slice
url = "https://raw.githubusercontent.com/arisckm/vigilant-system/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Apply baseline scoring rule
# Example rule: Score higher if impressions are high and trend is down
median_impressions = df['impressions_90d'].median()
df['score'] = 0.0
df.loc[(df['impressions_90d'] > median_impressions) & (df['trend_direction'] == 'down'), 'score'] = 0.9
df.loc[df['trend_direction'] == 'down', 'score'] += 0.4

# Assign reason code and action label
df['reason_code'] = 'REFRESH_STALE_DECLINING_TRAFFIC'
df['action_label'] = 'prioritize_refresh'

# Sort queue by score descending
ranked_queue = df.sort_values(by='score', ascending=False)

# 3. Write ranked queue to CSV
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)
print(f"Ranked queue successfully written to {output_path} with {len(ranked_queue)} rows.")

# Preview top 5
display(ranked_queue[['content_id', 'impressions_90d', 'trend_direction', 'score', 'action_label']].head(5))

Ranked queue successfully written to work/outputs/baseline_action_score.csv with 30000 rows.


,content_id,impressions_90d,trend_direction,score,action_label
29984,content_a6568a7c07d7,1460,down,1.3,prioritize_refresh
29989,content_e859812ce999,29760,down,1.3,prioritize_refresh
9,content_c27558df2b0c,1240,down,1.3,prioritize_refresh
16263,content_0a043365efdb,1306,down,1.3,prioritize_refresh
16264,content_7299ec98cd1e,1042,down,1.3,prioritize_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# Display top 10 items for review with what would make them wrong
top_10 = ranked_queue.head(10).copy()
for idx, row in top_10.iterrows():
    print(f"Content ID: {row['content_id']} | Score: {row['score']} | Action: {row['action_label']}")
    print(f"  -> Why it's there: High historical traffic volume ({row['impressions_90d']} impressions) with a declining trend.")
    print(f"  -> What would make it wrong: If the traffic drop is due to seasonal content timing rather than content staleness.\n")


Content ID: content_a6568a7c07d7 | Score: 1.3 | Action: prioritize_refresh
  -> Why it's there: High historical traffic volume (1460 impressions) with a declining trend.
  -> What would make it wrong: If the traffic drop is due to seasonal content timing rather than content staleness.

Content ID: content_e859812ce999 | Score: 1.3 | Action: prioritize_refresh
  -> Why it's there: High historical traffic volume (29760 impressions) with a declining trend.
  -> What would make it wrong: If the traffic drop is due to seasonal content timing rather than content staleness.

Content ID: content_c27558df2b0c | Score: 1.3 | Action: prioritize_refresh
  -> Why it's there: High historical traffic volume (1240 impressions) with a declining trend.
  -> What would make it wrong: If the traffic drop is due to seasonal content timing rather than content staleness.

Content ID: content_0a043365efdb | Score: 1.3 | Action: prioritize_refresh
  -> Why it's there: High historical traffic volume (1306 impre

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# 4. Weak picks + leakage check
print("--- Leakage & Weak Picks Audit ---")
print("Features used: impressions_90d, trend_direction (both based on past 90-day history)")
print("Are future window variables present? False")
print("Are label-derived columns present in score inputs? False")
print("Audit status: Clean. Baseline rule relies strictly on past historical aggregates.")

--- Leakage & Weak Picks Audit ---
Features used: impressions_90d, trend_direction (both based on past 90-day history)
Are future window variables present? False
Are label-derived columns present in score inputs? False
Audit status: Clean. Baseline rule relies strictly on past historical aggregates.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.